# Global SOC Digital Soil Mapping — Pipeline Overview

**Goal**: Build an ML-ready dataset of soil organic carbon (SOC) observations at GlobalSoilMap standard depths by pooling six open-source global soil databases.

## Data sources

| Source | Region | Profiles | Depths | SOC unit |
|--------|--------|----------|--------|-----------|
| **WoSIS** (ISRIC) | Global | ~196 000 | Variable | g/kg |
| **ISCN** v3 | Global | ~71 000 | Variable | % → g/kg |
| **LUCAS** Topsoil | EU | ~40 000 | 0–20 cm | g/kg |
| **NCSCD** v2 | Arctic/Boreal | ~3 600 | Variable | g/kg |
| **RaCA** | CONUS | ~58 000 | 0–100 cm | g/kg |
| **AfSIS-I** | Sub-Saharan Africa | ~3 600 | 0–20, 20–50 cm | g/kg |

## Standard target depths (GlobalSoilMap)
`0–5 | 5–15 | 15–30 | 30–60 | 60–100 | 100–200 cm`

## Pipeline workflow
```
Collectors → Unit conversion → Depth harmonisation (EAQ spline) → QC → Covariates → ML dataset
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
print("Environment ready.")

Environment ready.


## 1  Run the full pipeline

The first run will download ~10–20 GB of raw data.  
Subsequent runs reuse the cache in `data/raw/`.

In [2]:
from soc_dsm.pipeline import DSMPipeline

pipeline = DSMPipeline(
    raw_dir        = "../data/raw",
    harmonized_dir = "../data/harmonized",
    ml_dir         = "../data/ml_ready",
    covariate_dir  = "../data/covariates",
    spatial_dedup_m = 100.0,   # merge profiles within 100 m
    drop_flagged    = False,   # keep flagged rows (inspect them)
)

# Disable covariate fetching for a quick first test:
df = pipeline.run(
    sources=["WoSIS", "ISCN", "LUCAS", "RaCA", "NCSCD", "AfSIS"],
    force_download    = False,
    attach_covariates = False,   # set True to also fetch env. predictors
)

16:58:58 [INFO] soc_dsm.collectors.base_collector — [WoSIS] Downloading raw data …
16:59:00 [ERROR] soc_dsm.pipeline — [Pipeline] WoSIS failed: 404 Client Error: NOT FOUND for url: https://zenodo.org/records/7799781/files/wosis_latest_profiles.tsv?download=1
Traceback (most recent call last):
  File "/Users/mipl/Documents/Soil Carbon/SoC Modelling/soc_dsm/pipeline.py", line 116, in run
    df_raw = collector.collect(force_download=force_download)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mipl/Documents/Soil Carbon/SoC Modelling/soc_dsm/collectors/base_collector.py", line 35, in collect
    self.download()
  File "/Users/mipl/Documents/Soil Carbon/SoC Modelling/soc_dsm/collectors/wosis_collector.py", line 41, in download
    self._download_file(_PROFILES_URL, self._profiles_path)
  File "/Users/mipl/Documents/Soil Carbon/SoC Modelling/soc_dsm/collectors/base_collector.py", line 80, in _download_file
    resp.raise_for_status()
  File "/Users/mipl/minic

RuntimeError: No data collected — check network access and source availability.

## 2  Dataset overview

In [ ]:
print(f"Shape : {df.shape}")
df.head(3)

In [ ]:
# Missingness report
miss = (df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)
print("Missing (%) per column:")
print(miss[miss > 0].to_string())

## 3  SOC distribution by depth and source

In [ ]:
from soc_dsm.harmonizer.schema import STANDARD_DEPTHS

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
axes = axes.flatten()

for ax, (u, l) in zip(axes, STANDARD_DEPTHS):
    subset = df[(df["upper_depth"] == u) & (df["lower_depth"] == l)]
    soc = pd.to_numeric(subset["soc_g_per_kg"], errors="coerce").dropna()
    if soc.empty:
        ax.set_visible(False)
        continue
    soc_clipped = soc.clip(0, 200)   # cap display at 200 g/kg for readability
    ax.hist(soc_clipped, bins=80, color="#4c7c3f", edgecolor="none", alpha=0.8)
    ax.set_title(f"{u}–{l} cm  (n={len(soc):,})", fontsize=10)
    ax.set_xlabel("SOC (g kg⁻¹)")
    ax.set_ylabel("Count")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

fig.suptitle("SOC distribution at GlobalSoilMap standard depths", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../data/ml_ready/soc_distribution_by_depth.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Records per source
if "source_db" in df.columns:
    counts = df.groupby("source_db").size().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind="bar", ax=ax, color="#4c7c3f", edgecolor="white")
    ax.set_xlabel("")
    ax.set_ylabel("Horizon-depth slice records")
    ax.set_title("Records per source database")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 4  Global spatial coverage

In [ ]:
# Plot the 0-5 cm slice only (one point per profile)
top = df[(df["upper_depth"] == 0) & (df["lower_depth"] == 5)].copy()
top["longitude"] = pd.to_numeric(top["longitude"], errors="coerce")
top["latitude"]  = pd.to_numeric(top["latitude"],  errors="coerce")
top = top.dropna(subset=["longitude", "latitude", "soc_g_per_kg"])

fig, ax = plt.subplots(figsize=(16, 7))
sc = ax.scatter(
    top["longitude"], top["latitude"],
    c=top["soc_g_per_kg"].clip(0, 100),
    cmap="YlOrBr", s=1, alpha=0.5, vmin=0, vmax=100
)
plt.colorbar(sc, ax=ax, label="SOC g kg⁻¹ (capped at 100)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Global SOC observations — 0–5 cm  (n={len(top):,} profiles)")
plt.tight_layout()
plt.savefig("../data/ml_ready/global_soc_map_0-5cm.png", dpi=150, bbox_inches="tight")
plt.show()

## 5  QC flag summary

In [ ]:
if "qc_flags" in df.columns:
    all_flags = df["qc_flags"].dropna().str.split(",").explode()
    all_flags = all_flags[all_flags != ""]
    print("QC flag counts:")
    print(all_flags.value_counts().to_string())
else:
    print("No qc_flags column found.")

## 6  Prepare final ML feature matrix

Drop QC-flagged rows (optional) and select feature columns.

In [ ]:
# ---- Filter: only unflagged records with valid SOC ----
clean = df.copy()
if "qc_flags" in clean.columns:
    clean = clean[clean["qc_flags"].str.len() == 0]
clean = clean.dropna(subset=["soc_g_per_kg", "longitude", "latitude"])

print(f"Clean records: {len(clean):,}  (of {len(df):,} total)")

# ---- Feature columns ----
feature_cols = [
    "longitude", "latitude",
    "upper_depth", "lower_depth",
    "bulk_density_g_cm3", "coarse_fragments_pct",
    "ph_h2o", "clay_pct", "silt_pct", "sand_pct",
    "cec_cmol_kg", "total_n_g_per_kg",
    # add covariate columns here after attach_covariates=True:
    # "elevation_m", "chelsa_bio01", "chelsa_bio12", ...
]
target_col = "soc_g_per_kg"

available = [c for c in feature_cols if c in clean.columns]
X = clean[available]
y = clean[target_col]

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")
print(f"Target   : {target_col}  →  mean={y.mean():.1f}, median={y.median():.1f} g/kg")

In [ ]:
# Correlation heatmap of numeric features vs SOC
num_cols = [c for c in available
            if clean[c].dtype in ["float64", "float32", "int64", "int32"]]

corr = clean[num_cols + [target_col]].corr()[[target_col]].drop(target_col)
corr = corr.sort_values(target_col)

fig, ax = plt.subplots(figsize=(5, len(corr) * 0.5 + 1))
colors = ["#d73027" if v < 0 else "#4c7c3f" for v in corr[target_col]]
ax.barh(corr.index, corr[target_col], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson r with SOC")
ax.set_title("Feature correlations with SOC (g/kg)")
plt.tight_layout()
plt.savefig("../data/ml_ready/feature_correlation_soc.png", dpi=150, bbox_inches="tight")
plt.show()

## 7  Quick baseline model (Random Forest)

This is just a sanity check — the covariate-enriched dataset (Step 6 in `pipeline.run`) is needed for a production DSM model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

X_model = X.copy()
y_model = np.log1p(y)   # log-transform SOC (right-skewed)

rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   RandomForestRegressor(
        n_estimators=200, max_features="sqrt",
        min_samples_leaf=5, n_jobs=-1, random_state=42
    ))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_pipe, X_model, y_model, cv=cv,
                         scoring="r2", n_jobs=-1)

print(f"Random Forest 5-fold CV R² (log SOC): {scores.mean():.3f} ± {scores.std():.3f}")
print("(Low R² expected at this stage — covariates not yet attached)")

---
## Next steps

1. **Re-run with `attach_covariates=True`** to fetch terrain (SRTM), climate (CHELSA), and SoilGrids legacy predictions.
2. **Feature engineering**: derive depth mid-point, C:N ratio, log-transform skewed predictors.
3. **Spatial cross-validation** (`sklearn-spatial-cv` or `spatial-blocks`) to avoid inflated scores from spatial autocorrelation.
4. **Advanced models**: Quantile Random Forest (uncertainty), Gradient Boosting (XGBoost/LightGBM), or 3D Convolutional Net treating depth as a sequence.
5. **Prediction grid**: rasterise covariates to 250 m × 250 m grid → predict SOC at each depth → stack into a 3-D SOC volume.
6. **Validation**: compare against GSOCmap v1.6 and SoilGrids v2 as independent benchmarks.
